<a href="https://colab.research.google.com/github/nirvana66649/huggingface_transformers_nlp_tasks/blob/text_classification/Machine_Reading_Comprehension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.1 MB/s eta 0:00:00


# **机器阅读理解（Machine Reading Comprehension, MRC）是自然语言处理（NLP）中的一个核心任务，其目标是让计算机像人一样理解一段自然语言文本，并基于此文本回答相关问题。这项任务在智能问答系统、搜索引擎、虚拟助手等领域有广泛应用**

抽取式阅读理解（Extractive MRC）

答案是文章中的一个连续或不连续的片段。

代表数据集：SQuAD、CMRC2018。

示例：

文本：张三是中国北京的一名程序员。

问题：张三住在哪里？

答案：北京*

对于MRC任务，最重要的就是数据的预处理，对于抽取式阅读理解，通常会把数据进行划分-->[CLS] 问题 tokens [SEP] 上下文 tokens [SEP], 这一步在tokenize中实现

In [2]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering,TrainingArguments,Trainer,DefaultDataCollator
from datasets import load_dataset
import evaluate

## Step1 数据集加载cmrc2018

In [3]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [3]:
ds = load_dataset("cmrc2018")
ds

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.85k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/3.37M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

test-00000-of-00001.parquet:   0%|          | 0.00/395k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10142 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3219 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1002 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 10142
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 3219
    })
    test: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 1002
    })
})

可以看到，数据集的结构：context是文本内容，question：是为问题， answer为答案和答案的起始位置

In [4]:
ds['train'][0]

{'id': 'TRAIN_186_QUERY_0',
 'context': '范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年被擢升为天主教河内总教区宗座署理；1994年被擢升为总主教，同年年底被擢升为枢机；2009年2月离世。范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生；童年时接受良好教育后，被一位越南神父带到河内继续其学业。范廷颂于1940年在河内大修道院完成神学学业。范廷颂于1949年6月6日在河内的主教座堂晋铎；及后被派到圣女小德兰孤儿院服务。1950年代，范廷颂在河内堂区创建移民接待中心以收容到河内避战的难民。1954年，法越战争结束，越南民主共和国建都河内，当时很多天主教神职人员逃至越南的南方，但范廷颂仍然留在河内。翌年管理圣若望小修院；惟在1960年因捍卫修院的自由、自治及拒绝政府在修院设政治课的要求而被捕。1963年4月5日，教宗任命范廷颂为天主教北宁教区主教，同年8月15日就任；其牧铭为「我信天主的爱」。由于范廷颂被越南政府软禁差不多30年，因此他无法到所属堂区进行牧灵工作而专注研读等工作。范廷颂除了面对战争、贫困、被当局迫害天主教会等问题外，也秘密恢复修院、创建女修会团体等。1990年，教宗若望保禄二世在同年6月18日擢升范廷颂为天主教河内总教区宗座署理以填补该教区总主教的空缺。1994年3月23日，范廷颂被教宗若望保禄二世擢升为天主教河内总教区总主教并兼天主教谅山教区宗座署理；同年11月26日，若望保禄二世擢升范廷颂为枢机。范廷颂在1995年至2001年期间出任天主教越南主教团主席。2003年4月26日，教宗若望保禄二世任命天主教谅山教区兼天主教高平教区吴光杰主教为天主教河内总教区署理主教；及至2005年2月19日，范廷颂因获批辞去总主教职务而荣休；吴光杰同日真除天主教河内总教区总主教职务。范廷颂于2009年2月22日清晨在河内离世，享年89岁；其葬礼于同月26日上午在天主教河内总教区总主教座堂举行。',
 'question': '范廷颂是什么时候被任为主教的？',
 'answers': {'text': ['1963年'], 'answer_start': [30]}}

## Step2 数据预处理（加载tokenzier并进行处理）

In [5]:
# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

off_mapping长什么样：



```
offset_mapping = [
    (start_char_0, end_char_0),
    (start_char_1, end_char_1),
    (start_char_2, end_char_2),
    ...
]

```



***将抽取式问答数据集中每条样本的“答案字符位置”转换为对应的“token 起止位置”，用于训练机器阅读理解模型***

In [6]:
def process_func(examples):
    # 对每一组 (question, context) 进行分词编码，返回 token 与原始文本的字符位置映射（offsets）
    tokenized_examples = tokenizer(
        text=examples["question"],              # 问题文本
        text_pair=examples["context"],          # 上下文文本
        return_offsets_mapping=True,            # 返回每个 token 在原始 context 中的起止字符位置
        max_length=384,                         # 最大长度限制
        truncation="only_second",               # 仅对 context 部分进行截断（问题通常较短）
        padding="max_length"                    # 不足 max_length 的填充
    )

    offset_mapping = tokenized_examples.pop("offset_mapping")  # 提取 offset 信息并从结果中移除,这里是每一个token对于的索引
    start_positions = []
    end_positions = []

    # 遍历每一个样本（已被 tokenizer 编码）
    for idx, offset in enumerate(offset_mapping):
        answer = examples["answers"][idx]  # 获取该样本的答案字典
        start_char = answer["answer_start"][0]  # 答案起始字符位置（context 中的字符偏移）
        end_char = start_char + len(answer["text"][0])  # 答案结束字符位置

        # 获取当前样本中 context 的 token 范围
        sequence_ids = tokenized_examples.sequence_ids(idx)  # 标记每个 token 属于哪个句子：0-question，1-context，None-padding
        context_start = sequence_ids.index(1)  # context 的第一个 token 的索引
        context_end = sequence_ids.index(None, context_start) - 1  # context 的最后一个 token 的索引

        # 检查答案是否完全落在当前 context 范围内（排除被截断的情况）
        if offset[context_end][1] < start_char or offset[context_start][0] > end_char:
            start_token_pos = 0  # 设为 0，表示答案不在这段中（一般训练时忽略）
            end_token_pos = 0
        else:
            # 从左向右找到包含 start_char 的 token
            token_id = context_start
            while token_id <= context_end and offset[token_id][0] < start_char:
              # 直到找到第一个其起始字符位置不小于答案起点的 token，也就是**“包含答案起点的 token”**
                token_id += 1
            start_token_pos = token_id

            # 从右向左找到包含 end_char 的 token
            token_id = context_end
            while token_id >= context_start and offset[token_id][1] > end_char:
                token_id -= 1
            end_token_pos = token_id

        # 将计算出的 token 起止位置 加入结果列表
        start_positions.append(start_token_pos)
        end_positions.append(end_token_pos)

    # 添加到最终的 tokenizer 输出结果中，作为监督标签
    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples




```
answer = {
    "text": ["人工智能"],
    "answer_start": [135]
}

start_char = answer["answer_start"][0]  # 135
end_char = start_char + len(answer["text"][0])  # 135 + 5 = 140

```



In [7]:
tokenized_ds = ds.map(process_func,batched=True,remove_columns=ds['train'].column_names)
tokenized_ds

Map:   0%|          | 0/10142 [00:00<?, ? examples/s]

Map:   0%|          | 0/3219 [00:00<?, ? examples/s]

Map:   0%|          | 0/1002 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 10142
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 3219
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 1002
    })
})

In [ ]:
print(tokenized_ds['train'][0])

{'input_ids': [101, 5745, 2455, 7563, 3221, 784, 720, 3198, 952, 6158, 818, 711, 712, 3136, 4638, 8043, 102, 5745, 2455, 7563, 3364, 3322, 8020, 8024, 8021, 8024, 1760, 1399, 924, 4882, 185, 5735, 4449, 8020, 8021, 8024, 3221, 6632, 1298, 5384, 7716, 1921, 712, 3136, 3364, 3322, 511, 9155, 2399, 6158, 818, 711, 712, 3136, 8039, 8431, 2399, 6158, 3091, 1285, 711, 1921, 712, 3136, 3777, 1079, 2600, 3136, 1277, 2134, 2429, 5392, 4415, 8039, 8447, 2399, 6158, 3091, 1285, 711, 2600, 712, 3136, 8024, 1398, 2399, 2399, 2419, 6158, 3091, 1285, 711, 3364, 3322, 8039, 8170, 2399, 123, 3299, 4895, 686, 511, 5745, 2455, 7563, 754, 9915, 2399, 127, 3299, 8115, 3189, 1762, 6632, 1298, 2123, 2398, 4689, 1921, 712, 3136, 1355, 5683, 3136, 1277, 1139, 4495, 8039, 4997, 2399, 3198, 2970, 1358, 5679, 1962, 3136, 5509, 1400, 8024, 6158, 671, 855, 6632, 1298, 4868, 4266, 2372, 1168, 3777, 1079, 5326, 5330, 1071, 2110, 689, 511, 5745, 2455, 7563, 754, 9211, 2399, 1762, 3777, 1079, 1920, 934, 6887, 7368, 213

## Step3 加载模型


In [8]:
model = AutoModelForQuestionAnswering.from_pretrained("hfl/chinese-macbert-base")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/412M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at hfl/chinese-macbert-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step4 配置模型训练参数和训练器

In [9]:
args = TrainingArguments(
    output_dir="models_for_qa",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=3
)


In [10]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=DefaultDataCollator()
)

<ipython-input-10-6af0fe73917e>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jinnyertha (jinnyertha-meta) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,1.387500,1.110552
2,0.935000,1.076957
3,0.621900,1.189535


TrainOutput(global_step=951, training_loss=1.116698927495758, metrics={'train_runtime': 1005.559, 'train_samples_per_second': 30.258, 'train_steps_per_second': 0.946, 'total_flos': 5962661340337152.0, 'train_loss': 1.116698927495758, 'epoch': 3.0})

## step5 模型评估和预测

In [12]:
from transformers import pipeline


In [15]:
mrc_pipe = pipeline("question-answering",model=model,tokenizer=tokenizer)
mrc_pipe(question="张三的工作是什么？",context="张三是中国北京的一名程序员。")

Device set to use cuda:0


{'score': 0.6532812118530273, 'start': 10, 'end': 13, 'answer': '程序员'}